In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="038INb0Av0p6eo4CVxxx")
project = rf.workspace("marcuss-workspace").project("utility-poles-kcumt-nt0d2")
version = project.version(2)
dataset = version.download("yolo26")
                      

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26mpt")

In [ ]:
# Train the YOLO model on the downloaded dataset using optimal parameters
results = model.train(
    data="Utility-Poles-2/data.yaml",
    epochs=100,
    imgsz=1280,
    batch=8,
    device=[0,1,2,3],  # Use GPU if available; adjust as needed
    workers=8,
    patience=50,
    optimizer='SGD',
    lr0=0.01,
    cos_lr=True,
    mosaic=1.0,
    mixup=0.15,
    degrees=10.0,
    project='utility-pole-training',
    name='yolo26m-utility-poles',
    verbose=True,
    pretrained=True,
    cache=True,
    save_period=10  # Save a checkpoint every 10 epochs for ensembling
)

In [ ]:
# Run inference on a single validation image and display the result

import cv2
from matplotlib import pyplot as plt

# Get a sample image path from the validation set
sample_img_path = "utility-poles-2/valid/images"
import os
sample_images = [f for f in os.listdir(sample_img_path) if f.endswith(".jpg") or f.endswith(".jpeg") or f.endswith(".png")]

if sample_images:
    img_path = os.path.join(sample_img_path, sample_images[0])

    # Run prediction with test-time augmentation for better accuracy
    results = model(img_path, augment=True)

    # Visualize the result
    annotated_img = results[0].plot()  # returns an array (BGR by default)

    # Convert from BGR to RGB for matplotlib
    annotated_img_rgb = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_img_rgb)
    plt.axis("off")
    plt.title("YOLO Inference Result on Sample Image")
    plt.show()
else:
    print("No sample images found in the validation directory.")